# Export dbt Marts to Excel

Loads every table in the `ecomm_marts` BigQuery dataset and writes each one to a
named sheet in a single Excel workbook. Sheets are sorted alphabetically and
columns are auto-fitted to content width for readability.

## Step 1 — Setup: Libraries and Configuration

In [1]:
import sys
sys.path.append('..')
import config
import importlib
importlib.reload(config)

import pandas as pd
from google.cloud import bigquery
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import os

# ── Configuration ──────────────────────────────────────────────────
BQ_PROJECT_ID = config.PROJECT_ID
BQ_DATASET    = 'ecomm_marts'

# Output folder — one level up from notebooks/, into the outputs/ subfolder
OUTPUT_DIR  = os.path.join('..', 'outputs')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f'{BQ_DATASET}_export.xlsx')

os.makedirs(OUTPUT_DIR, exist_ok=True)  # create outputs/ if it doesn't exist yet

print(f'Project : {BQ_PROJECT_ID}')
print(f'Dataset : {BQ_DATASET}')
print(f'Output  : {os.path.abspath(OUTPUT_PATH)}')


Credentials set from /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_ecomm/credentials/portfolio-thelook-92bf870246df.json
Project ID: portfolio-thelook
Credentials set from /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_ecomm/credentials/portfolio-thelook-92bf870246df.json
Project ID: portfolio-thelook
Project : portfolio-thelook
Dataset : ecomm_marts
Output  : /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_ecomm/outputs/ecomm_marts_export.xlsx


## Step 2 — Connect to BigQuery

In [2]:
client = bigquery.Client(project=BQ_PROJECT_ID)
print('BigQuery client ready.')


BigQuery client ready.


## Step 3 — Discover Tables

Queries `INFORMATION_SCHEMA` to get the full list of tables in the dataset.
Falls back to a hardcoded list if the query fails.

In [3]:
_FALLBACK_TABLES = [
    'dim_distribution_centers', 'dim_products', 'dim_users',
    'fct_funnel', 'fct_order_items', 'fct_orders'
]

def get_table_list():
    query = (
        f'SELECT table_name FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.INFORMATION_SCHEMA.TABLES`'
        f" WHERE table_type = 'BASE TABLE' ORDER BY table_name"
    )
    try:
        tables = client.query(query).to_dataframe()['table_name'].tolist()
        if not tables:
            print('Warning: INFORMATION_SCHEMA returned 0 tables — using fallback list.')
            return _FALLBACK_TABLES
        return tables
    except Exception as e:
        print(f'Warning: Could not query INFORMATION_SCHEMA ({e}) — using fallback list.')
        return _FALLBACK_TABLES

tables = get_table_list()
print(f'Found {len(tables)} tables (alphabetical order):')
for t in tables:
    print(f'  {t}')


/Users/marcalexander/projects/evidence_project/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Found 7 tables (alphabetical order):
  dim_distribution_centers
  dim_products
  dim_users
  fct_funnel
  fct_inventory_xact
  fct_order_items
  fct_orders


## Step 4 — Helper Functions

**`strip_timezones(df)`** — Excel cannot write timezone-aware datetime values.
This function finds every datetime column in the DataFrame and removes its timezone
info, keeping the underlying UTC timestamp value unchanged.

**`load_table(table_name)`** — Loads a full BigQuery table into a DataFrame.

**`style_sheet(ws, df)`** — Applies a styled header row and auto-fits column widths.

In [4]:
def strip_timezones(df):
    """
    Convert all timezone-aware datetime columns to timezone-naive.
    BigQuery returns timestamps in UTC; this removes the tz label so
    openpyxl can write the values without error.
    """
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            if hasattr(df[col].dt, 'tz') and df[col].dt.tz is not None:
                df[col] = df[col].dt.tz_localize(None)
    return df


def load_table(table_name):
    query = f'SELECT * FROM `{BQ_PROJECT_ID}.{BQ_DATASET}.{table_name}`'
    return client.query(query).to_dataframe()


def style_sheet(ws, df):
    """Apply header styling and auto-fit column widths."""
    header_fill  = PatternFill(fill_type='solid', fgColor='1F4E79')  # dark navy
    header_font  = Font(bold=True, color='FFFFFF', size=11)
    header_align = Alignment(horizontal='center', vertical='center', wrap_text=False)

    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font      = header_font
        cell.fill      = header_fill
        cell.alignment = header_align

        # Auto-fit: max of header length and longest value in the column (capped at 60)
        max_len = max(
            len(str(col_name)),
            df[col_name].astype(str).str.len().max() if len(df) > 0 else 0
        )
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 2, 60)

    # Freeze the header row so it stays visible while scrolling
    ws.freeze_panes = 'A2'


print('Helper functions defined: strip_timezones, load_table, style_sheet')


Helper functions defined: strip_timezones, load_table, style_sheet


## Step 5 — Export to Excel

Loads each table, strips timezone info from datetime columns, then writes
to a named sheet. Prints progress as each sheet completes.

In [5]:
results = []  # track row counts for the summary

with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for table_name in tables:
        print(f'Loading {table_name}...', end=' ')
        try:
            df = load_table(table_name)
            df = strip_timezones(df)          # remove tz before writing to Excel
            sheet_name = table_name[:31]      # Excel sheet names max 31 chars
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            style_sheet(writer.sheets[sheet_name], df)
            results.append((table_name, len(df), len(df.columns), 'OK'))
            print(f'{len(df):,} rows, {len(df.columns)} columns — done')
        except Exception as e:
            results.append((table_name, 0, 0, f'ERROR: {e}'))
            print(f'FAILED — {e}')

print(f'\nWorkbook saved to: {os.path.abspath(OUTPUT_PATH)}')

# Summary table
print(f'\n{"="*65}')
print(f'Export Summary')
print(f'{"="*65}')
print(f'{"Table":<35} {"Rows":>10} {"Columns":>8}  Status')
print(f'{"-"*65}')
total_rows = 0
for table_name, rows, cols, status in results:
    print(f'{table_name:<35} {rows:>10,} {cols:>8}  {status}')
    total_rows += rows
print(f'{"-"*65}')
print(f'{"TOTAL":<35} {total_rows:>10,}')
print(f'{"="*65}')


Loading dim_distribution_centers... 10 rows, 5 columns — done
Loading dim_products... 

/Users/marcalexander/projects/evidence_project/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


29,120 rows, 16 columns — done
Loading dim_users... 

/Users/marcalexander/projects/evidence_project/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


100,000 rows, 19 columns — done
Loading fct_funnel... 

...

487,718 rows, 12 columns — done
Loading fct_order_items... 

/Users/marcalexander/projects/evidence_project/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


180,665 rows, 15 columns — done
Loading fct_orders... 

/Users/marcalexander/projects/evidence_project/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


124,919 rows, 20 columns — done

Workbook saved to: /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_ecomm/outputs/ecomm_marts_export.xlsx

Export Summary
Table                                     Rows  Columns  Status
-----------------------------------------------------------------
dim_distribution_centers                    10        5  OK
dim_products                            29,120       16  OK
dim_users                              100,000       19  OK
fct_funnel                             680,665       13  OK
fct_inventory_xact                     487,718       12  OK
fct_order_items                        180,665       15  OK
fct_orders                             124,919       20  OK
-----------------------------------------------------------------
TOTAL                                1,603,097


## Step 6 — Open Workbook

In [6]:
import subprocess, sys

abs_path = os.path.abspath(OUTPUT_PATH)
if sys.platform == 'darwin':
    subprocess.run(['open', abs_path])
elif sys.platform == 'win32':
    os.startfile(abs_path)
else:
    subprocess.run(['xdg-open', abs_path])
